# Coding Agent Lab

In this lab we build a tiny coding agent that repairs the deliberately broken project in `broken-repo/`.

The agent will use four tools:

- `inspect_files` to see the repository shape
- `read_file` to inspect source and tests
- `apply_patch` to edit files with a unified diff
- `run_tests` to verify the repair

Keep the design intentionally small. The point is to see the agent loop clearly: observe, choose one action, run a tool, feed the result back, and stop only when the tests pass.


## 0. Setup

The lab uses Ollama through the OpenAI-compatible local endpoint. Before running the notebook, start Ollama and pull the small coding model:

```bash
ollama pull qwen2.5-coder:1.5b
```

Install the Python client if needed:

```bash
python3 -m pip install -U openai
```


In [ ]:
from __future__ import annotations

import json
import importlib
import re
import subprocess
import textwrap
from pathlib import Path

import utils

utils = importlib.reload(utils)
AgentTrace = utils.AgentTrace
display_agent_trace = utils.display_agent_trace
load_llm = utils.load_llm
print_raw_agent_step = utils.print_raw_agent_step

LAB_ROOT = Path.cwd().resolve()
if LAB_ROOT.name == "broken-repo":
    LAB_ROOT = LAB_ROOT.parent

REPO = LAB_ROOT / "broken-repo"
TEST_COMMAND = ["python3", "-m", "unittest", "discover", "-s", "tests"]

assert REPO.exists(), f"Could not find broken-repo at {REPO}"

print(f"Lab root: {LAB_ROOT}")
print(f"Target repo: {REPO}")


## 1. Load the local model

`load_llm()` returns a tiny chat wrapper. It raises a helpful error if Ollama is not reachable or the model has not been pulled yet.


In [ ]:
llm = load_llm(model='gemma4:latest', temperature=0.0, max_tokens=900)
print(llm("Reply with exactly: ready", max_tokens=10))


## 2. Capture the failing baseline

A coding agent needs an observation before it can reason. The first observation is the test output from the broken project.


In [ ]:
def run_tests() -> str:
    """Run the exercise test suite and return a compact text observation."""
    result = subprocess.run(
        TEST_COMMAND,
        cwd=REPO,
        check=False,
        capture_output=True,
        text=True,
    )
    command = " ".join(TEST_COMMAND)
    return textwrap.dedent(
        f"""
        $ {command}
        exit code: {result.returncode}

        STDOUT:
        {result.stdout.rstrip() or "(empty)"}

        STDERR:
        {result.stderr.rstrip() or "(empty)"}
        """
    ).strip()


baseline = run_tests()
print(baseline)


## 3. Add basic tools

These tools are deliberately plain Python functions. The path guard keeps the agent inside `broken-repo/`, which makes mistakes less expensive while still letting it inspect and edit the target project.


In [ ]:
IGNORED_DIRS = {".git", "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache"}
REPO_ROOT = REPO.resolve()


def safe_repo_path(path="."):
    """Resolve a path under broken-repo and reject paths outside it."""
    candidate = (REPO_ROOT / Path(path)).resolve()
    try:
        candidate.relative_to(REPO_ROOT)
    except ValueError as exc:
        raise ValueError(f"Path escapes broken-repo: {path}") from exc
    return candidate


def inspect_files(root: str = ".", max_depth: int = 3) -> str:
    """List files and directories under broken-repo."""
    base = safe_repo_path(root)
    if not base.exists():
        raise FileNotFoundError(root)

    rows: list[str] = []
    for path in sorted(base.rglob("*")):
        rel = path.relative_to(REPO_ROOT)
        if any(part in IGNORED_DIRS for part in rel.parts):
            continue
        if len(rel.parts) > max_depth:
            continue
        suffix = "/" if path.is_dir() else ""
        rows.append(f"{rel}{suffix}")
    return "\n".join(rows) or "(no files)"


def read_file(path, start_line=1, end_line=None, max_chars=6000):
    """Read a file from broken-repo with line numbers."""
    target = safe_repo_path(path)
    if not target.is_file():
        raise FileNotFoundError(path)

    file_lines = target.read_text(encoding="utf-8").splitlines()
    start = max(1, int(start_line))
    end = len(file_lines) if end_line is None else min(len(file_lines), int(end_line))
    selected = [f"{number:4}: {line}" for number, line in enumerate(file_lines[start - 1:end], start)]
    output = "\n".join(selected)
    if len(output) > max_chars:
        output = output[:max_chars] + "\n... [truncated]"
    return output


def apply_patch(patch: str) -> str:
    """Apply a unified diff relative to broken-repo."""
    return utils.apply_unified_patch(REPO, patch)


In [ ]:
print(inspect_files())


In [ ]:
print(read_file("tests/test_store.py"))


## 4. Define the agent prompt

This is a text-only tool protocol. The model must respond with exactly one JSON action. The notebook parses that action, runs the matching Python function, and sends the observation back to the model.


In [ ]:
AGENT_SYSTEM_PROMPT = """
You are a small coding agent in a teaching lab.

Your task is to repair the Python project in broken-repo so that its tests pass.
You can use exactly one tool per turn by replying with exactly one JSON object.
Do not include Markdown fences, commentary, or extra text outside the JSON object.

Available tools:
1. inspect_files
   args: {"root": ".", "max_depth": 3}
   Use this to understand the repository layout.

2. read_file
   args: {"path": "tasklet/store.py", "start_line": 1, "end_line": null}
   Use this before editing a file.

3. apply_patch
   args: {"patch": "unified diff here"}
   Use a minimal unified diff. Paths must be relative to broken-repo, for example tasklet/store.py. Include enough context lines for the patch tool to locate the change.

4. run_tests
   args: {}
   Use this after applying a patch.

5. finish
   args: {"summary": "what changed and why"}
   Use this only after the tests pass.

Response schema:
{
  "note": "brief reason for the selected tool",
  "tool": "one of inspect_files, read_file, apply_patch, run_tests, finish",
  "args": { }
}

Work carefully:
- Start from the test failure.
- Read the relevant tests and implementation before patching.
- Make the smallest code change that explains the failure.
- After every patch, run the tests.
""".strip()


## 5. Build the agent loop

The loop is short on purpose. Most real coding-agent systems are variations on this pattern with better parsing, richer tools, memory management, and stronger safety checks. The trace display lives in `utils.py`, so this cell can focus on agent control flow.


In [ ]:
TOOLS = {
    "inspect_files": inspect_files,
    "read_file": read_file,
    "apply_patch": apply_patch,
    "run_tests": run_tests,
}


def parse_action(reply: str) -> dict:
    """Parse the model's JSON action, with a small recovery for stray text."""
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", reply, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def call_tool(action: dict) -> str:
    tool_name = action.get("tool")
    args = action.get("args") or {}

    if tool_name == "finish":
        return json.dumps(args)
    if tool_name not in TOOLS:
        return f"ERROR: unknown tool {tool_name!r}"
    if not isinstance(args, dict):
        return "ERROR: args must be a JSON object"

    try:
        return TOOLS[tool_name](**args)
    except Exception as exc:
        return f"ERROR: {type(exc).__name__}: {exc}"


def maybe_show_trace(trace, show_trace=True, show_raw=False, final=False):
    if show_trace and (final or not show_raw):
        display_agent_trace(trace, clear=not show_raw)


def run_agent(max_steps=8, show_trace=True, show_raw=False):
    trace = AgentTrace()
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                "Repair broken-repo. Here is the initial failing test output:\n\n"
                f"{baseline}\n\nChoose the next tool action."
            ),
        },
    ]

    for step in range(1, max_steps + 1):
        reply = llm.chat(messages, max_tokens=1200, temperature=0.0)

        try:
            action = parse_action(reply)
        except Exception as exc:
            observation = f"ERROR: could not parse a JSON action: {exc}. Reply with exactly one JSON object."
            action = {"note": "Parser could not read the model response.", "tool": "parser_error", "args": {}}
            trace.add(step, action, observation, reply)
            if show_raw:
                print_raw_agent_step(step, reply, action["tool"], observation)
            maybe_show_trace(trace, show_trace=show_trace, show_raw=show_raw)
            messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": f"Observation:\n{observation}"})
            continue

        tool_name = action.get("tool")
        if tool_name == "finish":
            summary = action.get("args", {}).get("summary", "")
            trace.add(step, action, summary, reply)
            if show_raw:
                print_raw_agent_step(step, reply, tool_name, summary)
            maybe_show_trace(trace, show_trace=show_trace, show_raw=show_raw, final=True)
            return {"action": action, "trace": trace}

        observation = call_tool(action)
        trace.add(step, action, observation, reply)
        if show_raw:
            print_raw_agent_step(step, reply, tool_name, observation)
        maybe_show_trace(trace, show_trace=show_trace, show_raw=show_raw)

        messages.append({"role": "assistant", "content": json.dumps(action)})
        messages.append({"role": "user", "content": f"Observation:\n{observation}\n\nChoose the next tool action."})

    maybe_show_trace(trace, show_trace=show_trace, show_raw=show_raw, final=True)
    return {"action": None, "trace": trace}


## 6. Run the agent

This cell edits files under `broken-repo/`. After it finishes, inspect the patch and rerun the tests manually if you want a second check. Set `show_raw=True` to print the original unformatted step-by-step transcript.


In [ ]:
result = run_agent(max_steps=20, show_raw=False)
result["action"]


## 7. Discussion prompts

Once the agent works, try changing one thing at a time:

- What happens if the prompt does not require reading files before patching?
- What happens if `apply_patch` is removed and the model can only suggest changes?
- How would you make the patch parser more robust?
- How would you prevent the agent from editing tests?
- What extra tool would make the agent more useful without making the lab too complex?
